# Dataset Preparation


In [ ]:
import os
import numpy as np
import rasterio
import cv2
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, Model

Data Preprocessing

In [ ]:
# Point this to your dataset in Drive
BASE = '/content/drive/MyDrive/Solar_Potential_Project/dataset'
IMG_DIRS = {
    'train':   os.path.join(BASE, 'train'),
    'val':     os.path.join(BASE, 'val'),
    'test':    os.path.join(BASE, 'test'),
}
MASK_DIRS = {
    'train':   os.path.join(BASE, 'train_labels'),
    'val':     os.path.join(BASE, 'val_labels'),
    'test':    os.path.join(BASE, 'test_labels'),
}


In [ ]:
def load_tiff_rgb(path):
    """Read a GeoTIFF (H,W,3) and normalize to [0,1]."""
    with rasterio.open(path) as src:
        img = src.read([1,2,3]).transpose(1,2,0)
    return img.astype(np.float32) / 255.0

def load_tiff_mask(path):
    """Read single-band mask where 255=building, 0=background -> binary."""
    with rasterio.open(path) as src:
        m = src.read(1)
    # convert 255→1, 0→0
    return (m == 255).astype(np.uint8)[..., np.newaxis]


In [ ]:
import os
import rasterio
import numpy as np

def tile_folder(in_dir, out_dir, patch_size=256):
    print(f"Tiling `{in_dir}` → `{out_dir}`")     # diagnostic log
    os.makedirs(out_dir, exist_ok=True)
    cnt_total = 0
    for fname in sorted(os.listdir(in_dir)):
        if not (fname.lower().endswith('.tif') or fname.lower().endswith('.tiff')):
            continue
        src_path = os.path.join(in_dir, fname)
        with rasterio.open(src_path) as src:
            arr = src.read().transpose(1,2,0)       # (H,W,C)
            H, W = arr.shape[:2]
            base = os.path.splitext(fname)[0]
            cnt = 0
            for y in range(0, H, patch_size):
                for x in range(0, W, patch_size):
                    tile = arr[y:y+patch_size, x:x+patch_size]
                    if tile.shape[0] != patch_size or tile.shape[1] != patch_size:
                        continue
                    out_path = os.path.join(out_dir, f"{base}_{cnt}.tif")
                    meta = src.meta.copy()
                    meta.update({
                        'height': patch_size,
                        'width': patch_size,
                        'transform': rasterio.Affine(
                            src.transform.a, 0, src.transform.c + x*src.transform.a,
                            0, src.transform.e, src.transform.f + y*src.transform.e
                        )
                    })
                    with rasterio.open(out_path, 'w', **meta) as dst:
                        dst.write(tile.transpose(2,0,1))
                    cnt += 1
            print(f"  {cnt} tiles from {fname}")
            cnt_total += cnt
    print(f"→ Created {cnt_total} tiles in `{out_dir}`\n")



# --- Make sure these dicts actually point at distinct folders! ---
IMG_DIRS = {
    'train': '/content/drive/MyDrive/Solar_Potential_Project/dataset/train',
    'val'  : '/content/drive/MyDrive/Solar_Potential_Project/dataset/val',
    'test' : '/content/drive/MyDrive/Solar_Potential_Project/dataset/test',
}
MASK_DIRS = {
    'train': '/content/drive/MyDrive/Solar_Potential_Project/dataset/train_labels',
    'val'  : '/content/drive/MyDrive/Solar_Potential_Project/dataset/val_labels',
    'test' : '/content/drive/MyDrive/Solar_Potential_Project/dataset/test_labels',
}


# Tile everything
for split in ['train','val','test']:
    tile_folder(IMG_DIRS[split],  f"/content/{split}_img_tiles")
    tile_folder(MASK_DIRS[split], f"/content/{split}_mask_tiles")


In [ ]:
def build_dataset(img_dir, mask_dir):
    imgs = sorted(os.listdir(img_dir))
    masks = sorted(os.listdir(mask_dir))
    X, Y = [], []
    for im, mk in zip(imgs, masks):
        X.append(load_tiff_rgb(os.path.join(img_dir, im)))
        Y.append(load_tiff_mask(os.path.join(mask_dir, mk)))
    return np.stack(X,0), np.stack(Y,0)

X_train, Y_train = build_dataset("/content/train_img_tiles",  "/content/train_mask_tiles")
X_val,   Y_val   = build_dataset("/content/val_img_tiles",    "/content/val_mask_tiles")
X_test,  Y_test  = build_dataset("/content/test_img_tiles",   "/content/test_mask_tiles")

print("Train :", X_train.shape, Y_train.shape)
print("Val   :", X_val.shape,   Y_val.shape)
print("Test  :", X_test.shape,  Y_test.shape)